Data Grain and Target Population
**Grain:** Exactly one row per **delivered, reviewed, single-seller order** (`order_id`).
#### Inclusion Filters: Rationale and Trade-offs
1. **Delivered (`order_status == 'delivered'`)**
   - **Why it exists:** The treatment variable $T$ (late delivery) requires observing the actual delivery date (`order_delivered_customer_date`) relative to the estimated delivery date (`order_estimated_delivery_date`). "Late" is not mathematically or operationally defined for orders that were canceled, lost in transit, or still ongoing.
   - **What it costs:** Omits severe fulfillment failures (e.g., permanently lost or canceled parcels), meaning our conclusions do not apply to total non-delivery events.
2. **Single-Seller (`seller_count == 1`)**
   - **Why it exists:** Multi-seller orders introduce composite treatment exposure and ambiguous confounders (e.g., conflicting delivery routes, varying seller track records, split shipments, and differing dispatch handling times). Restricting to single-seller orders (which comprise $\approx 99\%$ of delivered orders) ensures every confounder (seller history, shipping distance, seller ratings) is well-defined and attributable.
   - **Population it does *not* speak for:** Multi-vendor basket orders ($\approx 1\%$). The results cannot be directly generalized to multi-seller shopping journeys where split-shipment dynamics could independently alter customer expectations and satisfaction.
3. **Reviewed (`review_score IS NOT NULL`)**
   - **Why it exists:** The primary outcome $Y$ (customer review score) is only observed when a review survey is submitted.
   - **What it costs (Selection Node in the DAG):** Conditioning on review completion ($S = 1$) introduces potential **selection / collider bias**. The propensity to leave a review is likely influenced by delivery friction ($T$) and unobserved customer temperaments or expectations ($U$). We quantify the review response rate in Section 4.4 and conduct sensitivity/bounding analyses in Question 9 to assess any bias introduced by this filter.

In [1]:
import sys
sys.path.append("..")

from src.causal import connect_raw, build_raw_table, prepare, PROCESSED

con = connect_raw()
raw = build_raw_table(con)
df, CONFOUNDERS = prepare(raw)
print(f"raw delivered: {len(raw):,}  |  analysis table: {len(df):,}  |  {len(CONFOUNDERS)} confounders")

# 1) Grain
assert df["order_id"].is_unique, "fan-out: an order appears twice"

# 2) No missing values in anything the models will see
assert df[CONFOUNDERS + ["is_late", "good_review"]].notna().all().all()

# 3) Treatment definition is consistent
assert (df["is_late"] == (df["days_late"] > 0)).all()

PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_parquet(PROCESSED / "analysis_table.parquet", index=False)

raw delivered: 96,203  |  analysis table: 93,832  |  50 confounders


In [2]:
raw["has_review"] = raw["review_score"].notna()
print(raw.groupby("is_late")["has_review"].agg(["mean", "size"]))

             mean   size
is_late                 
False    0.994536  89672
True     0.976573   6531


In [3]:
late = df[df["is_late"]]
print((late["review_created"] < late["delivered_date"]).mean(), "of late orders reviewed before arrival")

0.7441860465116279 of late orders reviewed before arrival
